# 03 — Évaluation LLM (prompts)

Compare les prompts (`youth_friendly`, `factual_strict`, `structured_citations`) :
- couverture des mots-clés attendus
- présence de citations
- refus honnête quand le contexte est vide

**Prérequis** : Qdrant + `OPENAI_API_KEY` (coût API non négligeable).

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from eval.llm_eval import run_llm_evaluation
from rag.prompts import list_prompts

print("Prompts :", list_prompts())

## Lancer l'évaluation

In [ ]:
# use_rag=True : retrieval + génération (recommandé)
report = run_llm_evaluation(use_rag=True)
print("best_prompt =", report["best_prompt"])
print("best_combined_score =", round(report["best_combined_score"], 3))

In [ ]:
rows = []
for r in report["results"]:
    rows.append({
        "prompt": r["prompt_name"],
        "avg_total": r["metrics"]["avg_total"],
        "refusal_score": r["refusal_score"],
        "combined_score": r["combined_score"],
    })
df_llm = pd.DataFrame(rows).sort_values("combined_score", ascending=False)
df_llm

In [ ]:
ax = df_llm.plot(
    x="prompt",
    y=["avg_total", "refusal_score", "combined_score"],
    kind="bar",
    figsize=(9, 4),
    rot=15,
)
ax.set_ylim(0, 1.05)
ax.set_title("Comparaison des prompts")
ax.set_ylabel("Score")
plt.tight_layout()
plt.show()

## Détail par question (meilleur prompt)

In [ ]:
best = next(
    r for r in report["results"] if r["prompt_name"] == report["best_prompt"]
)
details = pd.DataFrame([
    {
        "case_id": d["case_id"],
        "total": d["scores"]["total"],
        "keyword_coverage": d["scores"].get("keyword_coverage"),
        "citation": d["scores"].get("citation_score"),
        "preview": d["answer_preview"][:80],
    }
    for d in best["details"]
])
details